## Session 2 Hands-on 2: ISA Specification of FEATHER in TAIDL

### ISA: Data Model

(A) Streaming Buffer (`StrBuf`): stores the Inputs (IVN) and Outputs (OVN)

(B) Stationary Buffer (`StatBuf`): stores the Weights (WVN)

(C) 3 Set of Configuration Registers (`IVNConfig`, `WVNConfig`, `OVNConfig`) of type `LayoutConfig`

`LayoutConfig` is an alias for a tuple of 4 integers: `(order, M1, M0, J1)`

### ISA: Instruction Set

(A) Memory IO: `LoadIVN(addr)`, `LoadWVN(addr)`, `StoreOVN(addr)`

(B) Configuration: `SetIVN(LayoutConfig)`, `SetWVN(LayoutConfig)`, `SetOVN(LayoutConfig)`

(C) GEMM: `Execute()`

(D) SoftMax: `Softmax(LayoutConfig)`


### ISA: Semantics of `Execute()`

Data stored in `StrBuf` and `StatBuf` are in their physical view.  
`Execute()` performs the matrix multiplication over matrices in their logical view (2-D view).  
But the data is stored in `StrBuf` and `StatBuf` in their physical view.

In [ ]:
# Core Pseudocode:
def Execute():
    Input = logical(StrBuf, IVNConfig)
    Weight = logical(StatBuf, WVNConfig)
    Output = Input @ Weight
    StrBuf = physical(Output, OVNConfig)


### Semantics of `logical()` and `physical()`

Going back to our example of `f32[16x12]` input matrix

Logical View: `f32[16,12]`

Physical View: `f32[12,4,4]`

LayoutConfig: `(order, M1, M0, J1)`


In [ ]:
import jax.numpy as jnp
from jax.lax import reshape, transpose

AW = 4
VN_SIZE = 4

def order_to_perm(order: int) -> list:
    if order == 5:
        return [0, 1, 2, 3]  # M1, M0, J1, VN
    elif order == 4:
        return [0, 2, 1, 3]  # M1, J1, M0, VN
    elif order == 3:
        return [1, 0, 2, 3]  # M0, M1, J1, VN
    elif order == 2:
        return [1, 2, 0, 3]  # M0, J1, M1, VN
    elif order == 1:
        return [2, 0, 1, 3]  # J1, M1, M0, VN
    elif order == 0:
        return [2, 1, 0, 3]  # J1, M0, M1, VN
    else:
        raise ValueError(f"Invalid order: {order}. Must be in range [0-5].")

def physical(input: jnp.ndarray, order: int, M1: int, M0: int, J1: int) -> jnp.ndarray:
    assert input.shape == (16, 12), f"Expected input shape (16, 12), got {input.shape}"
    assert M1 * M0 == input.shape[0], f"M1 * M0 must equal input's first dimension (16), got M1={M1}, M0={M0}"
    assert J1 * VN_SIZE == input.shape[1], f"J1 * VN_SIZE must equal input's second dimension (12), got J1={J1}, VN_SIZE={VN_SIZE}"

    row_size = (M1 * M0 * J1) // (AW)

    # Semantics of physical transformation using tensor operators
    # TODO: Exercise
    pass

### Golden Reference Implementation

Let's test out against a golden reference implementation using FOR loops.


In [ ]:
import jax.numpy as jnp

AW = 4
VN_SIZE = 4

"""A golden implementation of the physical function for testing purposes"""
def physical_golden(input: jnp.ndarray, order: int, M1: int, M0: int, J1: int) -> jnp.ndarray:
    assert input.shape == (16, 12), f"Expected input shape (16, 12), got {input.shape}"
    assert M1 * M0 == input.shape[0], f"M1 * M0 must equal input's first dimension (16), got M1={M1}, M0={M0}"
    assert J1 * VN_SIZE == input.shape[1], f"J1 * VN_SIZE must equal input's second dimension (12), got J1={J1}, VN_SIZE={VN_SIZE}"

    row_size = (M1 * M0 * J1) // (AW)
    buffer = jnp.zeros((row_size, AW, VN_SIZE), dtype=input.dtype)

    if order == 5:
        row_index = 0
        col_index = 0
        for m1 in range(M1):
            for m0 in range(M0):
                for j1 in range(J1):
                    for j0 in range(VN_SIZE):
                        phy_row_index = (m1 * M0 * J1 + m0 * J1 + j1) // AW
                        phy_col_index = (m1 * M0 * J1 + m0 * J1 + j1) % AW
                        log_row_index = m1 * M0 + m0
                        log_col_index = j1 * VN_SIZE + j0
                        buffer = buffer.at[phy_row_index, phy_col_index, j0].set(input[log_row_index, log_col_index])
    elif order == 4:
        row_index = 0
        col_index = 0
        for m1 in range(M1):
            for j1 in range(J1):
                for m0 in range(M0):
                    for j0 in range(VN_SIZE):
                        phy_row_index = (m1 * M0 * J1 + j1 * M0 + m0) // AW
                        phy_col_index = (m1 * M0 * J1 + j1 * M0 + m0) % AW
                        log_row_index = m1 * M0 + m0
                        log_col_index = j1 * VN_SIZE + j0
                        buffer = buffer.at[phy_row_index, phy_col_index, j0].set(input[log_row_index, log_col_index])
    elif order == 3:
        row_index = 0
        col_index = 0
        for m0 in range(M0):
            for m1 in range(M1):
                for j1 in range(J1):
                    for j0 in range(VN_SIZE):
                        phy_row_index = (m0 * M1 * J1 + m1 * J1 + j1) // AW
                        phy_col_index = (m0 * M1 * J1 + m1 * J1 + j1) % AW
                        log_row_index = m1 * M0 + m0
                        log_col_index = j1 * VN_SIZE + j0
                        buffer = buffer.at[phy_row_index, phy_col_index, j0].set(input[log_row_index, log_col_index])
    elif order == 2:
        row_index = 0
        col_index = 0
        for m0 in range(M0):
            for j1 in range(J1):
                for m1 in range(M1):
                    for j0 in range(VN_SIZE):
                        phy_row_index = (m0 * J1 * M1 + j1 * M1 + m1) // AW
                        phy_col_index = (m0 * J1 * M1 + j1 * M1 + m1) % AW
                        log_row_index = m1 * M0 + m0
                        log_col_index = j1 * VN_SIZE + j0
                        buffer = buffer.at[phy_row_index, phy_col_index, j0].set(input[log_row_index, log_col_index])
    elif order == 1:
        row_index = 0
        col_index = 0
        for j1 in range(J1):
            for m1 in range(M1):
                for m0 in range(M0):
                    for j0 in range(VN_SIZE):
                        phy_row_index = (j1 * M1 * M0 + m1 * M0 + m0) // AW
                        phy_col_index = (j1 * M1 * M0 + m1 * M0 + m0) % AW
                        log_row_index = m1 * M0 + m0
                        log_col_index = j1 * VN_SIZE + j0
                        buffer = buffer.at[phy_row_index, phy_col_index, j0].set(input[log_row_index, log_col_index])
    elif order == 0:
        row_index = 0
        col_index = 0
        for j1 in range(J1):
            for m0 in range(M0):
                for m1 in range(M1):
                    for j0 in range(VN_SIZE):
                        phy_row_index = (j1 * M0 * M1 + m0 * M1 + m1) // AW
                        phy_col_index = (j1 * M0 * M1 + m0 * M1 + m1) % AW
                        log_row_index = m1 * M0 + m0
                        log_col_index = j1 * VN_SIZE + j0
                        buffer = buffer.at[phy_row_index, phy_col_index, j0].set(input[log_row_index, log_col_index])
    else:
      assert False, f"Invalid order {order}"

    return buffer

In [ ]:
# Test the physical function against the golden implementation
input = jnp.arange(16 * 12).reshape(16, 12)
order = 4
M1 = 8
M0 = 2
J1 = 3

output = physical(input, order, M1, M0, J1)
output_golden = physical_golden(input, order, M1, M0, J1)

if jnp.array_equal(output, output_golden):
    print("Test passed!")
else:
    print("Test failed!")
    print("Output:")
    print(output)
    print("Golden Output:")
    print(output_golden)

### Solution


In [ ]:
import jax.numpy as jnp
from jax.lax import reshape, transpose

AW = 4
VN_SIZE = 4

def order_to_perm(order: int) -> list:
    if order == 5:
        return [0, 1, 2, 3]  # M1, M0, J1, VN
    elif order == 4:
        return [0, 2, 1, 3]  # M1, J1, M0, VN
    elif order == 3:
        return [1, 0, 2, 3]  # M0, M1, J1, VN
    elif order == 2:
        return [1, 2, 0, 3]  # M0, J1, M1, VN
    elif order == 1:
        return [2, 0, 1, 3]  # J1, M1, M0, VN
    elif order == 0:
        return [2, 1, 0, 3]  # J1, M0, M1, VN
    else:
        raise ValueError(f"Invalid order: {order}. Must be in range [0-5].")

def physical(input: jnp.ndarray, order: int, M1: int, M0: int, J1: int) -> jnp.ndarray:
    assert input.shape == (16, 12), f"Expected input shape (16, 12), got {input.shape}"
    assert M1 * M0 == input.shape[0], f"M1 * M0 must equal input's first dimension (16), got M1={M1}, M0={M0}"
    assert J1 * VN_SIZE == input.shape[1], f"J1 * VN_SIZE must equal input's second dimension (12), got J1={J1}, VN_SIZE={VN_SIZE}"

    row_size = (M1 * M0 * J1) // (AW)

    # Semantics of physical transformation using tensor operators
    split = input.reshape(M1, M0, J1, VN_SIZE)
    permuted = transpose(split, order_to_perm(order))
    buffer = permuted.reshape(row_size, AW, VN_SIZE)

    return buffer